In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from model import FingerCountModel
import cv2
import time

In [11]:
!curl -o ../model/hand_landmarker.task -L "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task"

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  7.45M 100  7.45M   0      0 13.50M      0                              0


In [ ]:
cap = cv2.VideoCapture(0)
model = FingerCountModel("../model/hand_landmarker.task")

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error al leer la cámara.")
            break
            
        cv2.imshow('Prueba Local - C: Capturar | Q: Salir', frame)
        
        tecla = cv2.waitKey(1) & 0xFF
        
        # Si presiona 'c'
        if tecla == ord('c'):
            exito, buffer = cv2.imencode('.jpg', frame)
            
            image_bytes = buffer.tobytes()
            
            fingers_count = model.predict(image_bytes)
            print(f"Cantidad de dedos detectados: {fingers_count}")

        # Si presiona 'q'
        elif tecla == ord('q'):
            print("\nCerrando...")
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

I0000 00:00:1781028042.961736   81413 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1781028042.965714   81437 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 26.0.8), renderer: AMD Radeon Graphics (radeonsi, renoir, ACO, DRM 3.64, 7.0.10-201.fc44.x86_64)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1781028042.985549   81421 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781028042.997142   81424 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in "/home/alexc0dex/.local/lib/python3.14/site-packages/cv2/qt/plugins"
QFontDatabase: Cannot find font directory /home/alexc0dex/.local/lib/python3.14/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. De

Cantidad de dedos detectados: 5
Cantidad de dedos detectados: 4
Cantidad de dedos detectados: 3
Cantidad de dedos detectados: 2
Cantidad de dedos detectados: 1
Cantidad de dedos detectados: 2
Cantidad de dedos detectados: 1
Cantidad de dedos detectados: 2
Cantidad de dedos detectados: 3
Cantidad de dedos detectados: 2
Cantidad de dedos detectados: 3
Cantidad de dedos detectados: 4
Cantidad de dedos detectados: 5
Cantidad de dedos detectados: 4
Cantidad de dedos detectados: 5
Cantidad de dedos detectados: 1
Cantidad de dedos detectados: 2
Cantidad de dedos detectados: 3
Cantidad de dedos detectados: 4
Cantidad de dedos detectados: 1

Cerrando...


In [6]:
import cv2
import requests

URL_API = 'http://localhost:8000/predict'
ARCHIVO_FOTO = 'captura_para_api.jpg'

cap = cv2.VideoCapture(0)

real_label = 2

try:
    while True:
        ret, frame = cap.read()
        if not ret:
            print("Error al leer la cámara.")
            break
            
        cv2.imshow('Cliente API - C: Enviar | Q: Salir', frame)
        
        tecla = cv2.waitKey(1) & 0xFF
        
        if tecla == ord('c'):
            cv2.imwrite(ARCHIVO_FOTO, frame)

            try:
                with open(ARCHIVO_FOTO, 'rb') as f:
                    archivos = {'file': (ARCHIVO_FOTO, f, 'image/jpeg')}

                    datos = {"etiqueta_real": real_label}
                    
                    respuesta = requests.post(URL_API, files=archivos, data=datos)

                if respuesta.status_code == 200:
                    resultado = respuesta.json()
                    print(f"Conteo de la API: {resultado['fingers_count']} dedos.")
                    print(f"Mensaje del servidor: {resultado['message']}")
                    print(f"Ruta de guardado: {resultado['saved_path']}\n")
                else:
                    print(f"Error del servidor (Código {respuesta.status_code}): {respuesta.text}")
                    
            except requests.exceptions.ConnectionError:
                print("Error de conexión")
                
        # Si presiona 'q' (Salir)
        elif tecla == ord('q'):
            print("\nCerrando cliente de cámara...")
            break

finally:
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)

Conteo de la API: 2 dedos.
Mensaje del servidor: Prediction successful
Ruta de guardado: ./data/dataset_nao/2/2_img_20260609_172219_942454.jpg

Conteo de la API: 2 dedos.
Mensaje del servidor: Prediction successful
Ruta de guardado: ./data/dataset_nao/2/2_img_20260609_172223_062120.jpg

Conteo de la API: 2 dedos.
Mensaje del servidor: Prediction successful
Ruta de guardado: ./data/dataset_nao/2/2_img_20260609_172224_101949.jpg


Cerrando cliente de cámara...
